# Chapter 9 — Iterator & Composite Patterns
## *Well-Managed Collections*

---

## Iterator Pattern

**Intent:** Provide a way to access elements of a collection sequentially **without exposing its underlying representation**.

### OO Design Principle
- **Single Responsibility Principle:** A class should have only one reason to change.  
  Separate *traversal* from *collection management*.

### Book context
Two restaurants merge: one stores menu items in a `list`, the other in a `dict`.  
A waitress (`MenuComponent`) needs to iterate both uniformly.

In [ ]:
from __future__ import annotations
from abc import ABC, abstractmethod
from typing import Iterator

class MenuItem:
    def __init__(self, name: str, description: str,
                 vegetarian: bool, price: float):
        self.name        = name
        self.description = description
        self.vegetarian  = vegetarian
        self.price       = price

    def __repr__(self):
        v = "(v)" if self.vegetarian else ""
        return f"  {self.name}{v}, ${self.price:.2f} — {self.description}"


# Menu stored as a list
class PancakeHouseMenu:
    def __init__(self):
        self._items: list[MenuItem] = [
            MenuItem("K&B Pancake Breakfast", "Pancakes with eggs", True,  2.99),
            MenuItem("Regular Pancake Breakfast", "Pancakes with eggs and sausage", False, 2.99),
            MenuItem("Blueberry Pancakes", "Pancakes made with fresh blueberries", True, 3.49),
        ]

    def __iter__(self) -> Iterator[MenuItem]:
        return iter(self._items)   # Python lists already support iter


# Menu stored as a dict
class DinerMenu:
    def __init__(self):
        self._items: dict[str, MenuItem] = {
            "Vegetarian BLT": MenuItem("Vegetarian BLT", "Bacon with tomato on whole wheat", True, 2.99),
            "BLT":            MenuItem("BLT", "Bacon with tomato on whole wheat", False, 2.99),
            "Soup of the day":MenuItem("Soup of the day", "Soup and salad", False, 3.29),
        }

    def __iter__(self) -> Iterator[MenuItem]:
        return iter(self._items.values())  # dict values


class Waitress:
    def __init__(self, *menus):
        self._menus = menus

    def print_menu(self):
        for menu in self._menus:
            for item in menu:   # uniform iteration
                print(item)

    def print_vegetarian_menu(self):
        print("\n=== Vegetarian Items ===")
        for menu in self._menus:
            for item in menu:
                if item.vegetarian:
                    print(item)


waitress = Waitress(PancakeHouseMenu(), DinerMenu())
print("=== Full Menu ===")
waitress.print_menu()
waitress.print_vegetarian_menu()

## Custom iterator protocol

In [ ]:
class CountUp:
    """Custom iterator: yields integers from start to stop."""
    def __init__(self, start: int, stop: int):
        self._current = start
        self._stop    = stop

    def __iter__(self): return self

    def __next__(self) -> int:
        if self._current > self._stop:
            raise StopIteration
        val = self._current
        self._current += 1
        return val


print(list(CountUp(1, 5)))  # [1, 2, 3, 4, 5]

---

## Composite Pattern

**Intent:** Compose objects into **tree structures** to represent part-whole hierarchies.  
Composite lets clients treat individual objects (leaves) and compositions (composites) uniformly.

### Book context
A merged menu now needs sub-menus (e.g., a dessert menu inside the diner menu).  
Both `MenuItem` (leaf) and `Menu` (composite) implement the same `MenuComponent` interface.

In [ ]:
class MenuComponent(ABC):
    # Leaf methods
    @property
    def name(self) -> str:        raise NotImplementedError
    @property
    def description(self) -> str: raise NotImplementedError
    @property
    def price(self) -> float:     raise NotImplementedError
    @property
    def vegetarian(self) -> bool: raise NotImplementedError

    # Composite methods
    def add(self, c: MenuComponent):    raise NotImplementedError
    def remove(self, c: MenuComponent): raise NotImplementedError
    def get_child(self, i: int):        raise NotImplementedError

    @abstractmethod
    def print(self, depth: int = 0): ...


class MenuItemLeaf(MenuComponent):
    def __init__(self, name, description, vegetarian, price):
        self._name        = name
        self._description = description
        self._vegetarian  = vegetarian
        self._price       = price

    @property
    def name(self):        return self._name
    @property
    def description(self): return self._description
    @property
    def price(self):       return self._price
    @property
    def vegetarian(self):  return self._vegetarian

    def print(self, depth=0):
        v = "(v)" if self._vegetarian else ""
        print("  " * depth + f"  {self._name}{v}, ${self._price:.2f}")


class Menu(MenuComponent):
    def __init__(self, name: str, description: str):
        self._name        = name
        self._description = description
        self._children: list[MenuComponent] = []

    @property
    def name(self):        return self._name
    @property
    def description(self): return self._description

    def add(self, c):    self._children.append(c)
    def remove(self, c): self._children.remove(c)
    def get_child(self, i): return self._children[i]

    def print(self, depth=0):
        indent = "  " * depth
        print(f"\n{indent}[{self._name}] — {self._description}")
        for child in self._children:
            child.print(depth + 1)


# Build the tree
all_menus = Menu("ALL MENUS", "All menus combined")

pancake = Menu("Pancake House Menu", "Breakfast")
pancake.add(MenuItemLeaf("K&B Pancakes", "Pancakes with eggs", True, 2.99))
pancake.add(MenuItemLeaf("Waffles", "Waffles with blueberries", True, 3.59))

diner = Menu("Diner Menu", "Lunch")
diner.add(MenuItemLeaf("BLT", "Bacon on whole wheat", False, 2.99))
diner.add(MenuItemLeaf("Soup", "Soup and salad", False, 3.29))

dessert = Menu("Dessert Menu", "Have room for dessert?")
dessert.add(MenuItemLeaf("Apple Pie", "Apple pie with ice cream", True, 1.59))
diner.add(dessert)   # nested composite!

all_menus.add(pancake)
all_menus.add(diner)

all_menus.print()

## Interview cheat-sheet

| | Iterator | Composite |
|---|---|---|
| **Solves** | Uniform traversal of different collections | Tree structures / part-whole hierarchies |
| **Key interface** | `__iter__` / `__next__` | Shared interface for leaf and composite |
| **Python built-in** | All iterables, `itertools` | `pathlib.Path`, `xml.etree` |
| **SRP connection** | Separates iteration from collection | Clients don't distinguish leaf vs composite |

**Pattern family:** Behavioral (Iterator), Structural (Composite)